# LLM Legislativo Brasileiro — Piloto Colab / Drive

**Frente 08:** preflight, staging e persistência Colab/Drive.

Este notebook usa um dataset sintético para homologação do software. Ele não representa treinamento no corpus legislativo e a retomada criada no mesmo kernel é apenas um teste de integração; G5 requer repetir a restauração após encerrar e iniciar uma sessão Colab nova.

Antes de executar, abra este notebook a partir de um checkout limpo no commit imutável configurado em `LLM_REPO_REVISION`; monte o Drive, informe um shard real em `LLM_IO_BENCHMARK_FILE` e confirme GPU e escrita/leitura persistente no preflight. O saldo e os benefícios efetivos da conta devem ser conferidos no Colab autenticado.


In [ ]:
# [Célula 1] Configuração de Ambiente e Verificação de Dependências
import sys
import os
from pathlib import Path

# Garante que a raiz do repositório está no sys.path
REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
import legislm

print(f"Versão do legislm: {legislm.__version__}")
print(f"Versão do PyTorch: {torch.__version__}")
print(f"CUDA disponível no PyTorch: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")

In [ ]:
# A revisão do checkout deve ser fixada antes de qualquer execução.
import os, re, subprocess
expected_revision = os.environ.get('LLM_REPO_REVISION', '')
if not re.fullmatch(r'[0-9a-f]{40}', expected_revision):
    raise RuntimeError('Defina LLM_REPO_REVISION com o SHA completo do commit aprovado.')
actual_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_ROOT, text=True).strip()
if actual_revision != expected_revision or dirty:
    raise RuntimeError(f'Checkout divergente ou sujo: {actual_revision}; fixe o commit e limpe alterações locais.')
print(f'Código fixado em {actual_revision}')


## 2. Montagem do Google Drive e Termos de Serviço

> **Aviso Normativo:** Conforme `docs/COLAB.md`, créditos CCU do Google AI Pro são finitos e alocados por demanda; não pressuponha execução ininterrupta de 24h ou imunidade a desconexões efêmeras. O disco local da VM (`/content/`) não é expandido pela assinatura de armazenamento do Drive.

In [ ]:
# [Célula 2] Montagem Segura do Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    print("Google Drive montado com sucesso em /content/drive")
except ImportError:
    DRIVE_AVAILABLE = False
    print("Ambiente local detectado (fora do Colab). Utilizando caminhos locais de fallback.")

## 3. Pré-voo (Preflight): Auditoria de Hardware e Armazenamento

Execução da auditoria formal de GPU/VRAM/CUDA/RAM/disco/versões/precisão e teste de escrita/leitura/releitura com SHA-256 no Drive.

In [ ]:
# [Célula 3] Execução do Preflight
from legislm.colab.preflight import run_preflight

preflight_config = "configs/colab/pilot.yaml"
preflight_report = run_preflight(
    config=preflight_config,
    allow_cpu_fallback=False,  # G5 exige GPU alocada e Drive acessível
)

print(f"Status Geral do Preflight: {preflight_report['status']}")
if preflight_report['status'] != 'PASSED':
    raise RuntimeError('Preflight G5 reprovado: não iniciar treino Colab sem GPU/Drive aprovados.')
print(f"RAM Total: {preflight_report['hardware']['system_ram_gb']} GB")
print(f"Disco Local Livre: {preflight_report['hardware']['local_disk_free_gb']} GB")
if preflight_report['hardware']['gpu']['available']:
    gpu = preflight_report['hardware']['gpu']
    print(f"GPU: {gpu['device_name']} ({gpu['total_vram_gb']} GB VRAM, BF16={gpu['bf16_supported']})")
else:
    print("GPU: Não disponível (modo CPU fallback ativo para smoke).")

Benchmark diagnóstico do mesmo arquivo entre leitura pela origem montada e staging local, incluindo custo de cópia e hashes. Os tempos dependem do arquivo, cache e sessão; não generalize para o corpus. Informe um shard tokenizado ou archive representativo em `LLM_IO_BENCHMARK_FILE`.


In [ ]:
# [Célula 4] Benchmark de I/O e Demonstração de Staging
import os
from pathlib import Path
from legislm.colab.staging import benchmark_io, stage_file

benchmark_source = Path(os.environ['LLM_IO_BENCHMARK_FILE'])
if not benchmark_source.is_file():
    raise FileNotFoundError(f'Arquivo representativo não encontrado: {benchmark_source}')

io_results = benchmark_io(
    source_file_or_dir=benchmark_source,
    local_target_dir=Path('/content/llm_legislativo/benchmark_io'),
    num_reads=3,
)

print(f"Throughput leitura da origem: {io_results['direct_io']['avg_speed_mb_s']} MB/s")
print(f"Throughput leitura local staged: {io_results['local_staged_io']['avg_speed_mb_s']} MB/s")
print(f"Speedup observado nesta amostra: {io_results['speedup_factor']}x")
print(io_results['recommendation'])

# Materializa o mesmo shard no cache local configurado para consumo downstream.
staged_shard = Path('/content/llm_legislativo/staging') / benchmark_source.name
staging_result = stage_file(benchmark_source, staged_shard, verify_hashes=True, force_overwrite=True)
print(f"Shard staged: {staging_result['path']} ({staging_result['size_bytes']} bytes, SHA-256 {staging_result['sha256']})")


Instanciação explícita do Transformer Decoder PyTorch, sem Hugging Face Trainer. O conjunto sintético serve somente como smoke de integração; G5 histórico exige tokens liberados após G3/G4 e outra sessão Colab limpa.


In [ ]:
# [Célula 5] Execução do Treinador Piloto (Segmento 1)
from legislm.training.config import TrainingConfig
from legislm.training.data import create_synthetic_dataset
from legislm.training.engine import Trainer
from legislm.colab.persistence import GoogleDrivePersistenceBackend

# Carrega configuração padrão do piloto
cfg = TrainingConfig.from_yaml("configs/training/pilot.yaml")
cfg.device = "cuda" if torch.cuda.is_available() else "cpu"
cfg.persistence.backend = "drive"
cfg.persistence.runs_dir = "/content/drive/MyDrive/llm_legislativo/runs"
cfg.training.max_steps = 20
cfg.training.checkpoint_interval_steps = 5

# Inicializa backend de persistência com retenção dos 2 mais recentes + melhor checkpoint
backend = GoogleDrivePersistenceBackend(
    checkpoints_to_keep=2,
    keep_best=True,
    max_storage_bytes=int(os.environ['LLM_CHECKPOINT_QUOTA_BYTES']) if os.environ.get('LLM_CHECKPOINT_QUOTA_BYTES') else None,
    milestone_interval_steps=int(os.environ['LLM_CHECKPOINT_MILESTONE_STEPS']) if os.environ.get('LLM_CHECKPOINT_MILESTONE_STEPS') else None,
)

# Cria dataset representativo de treino e validação
train_ds = create_synthetic_dataset(
    num_docs=30,
    vocab_size=cfg.model.vocab_size,
    context_length=cfg.model.context_length,
    seed=cfg.training.seed,
)
eval_ds = create_synthetic_dataset(
    num_docs=10,
    vocab_size=cfg.model.vocab_size,
    context_length=cfg.model.context_length,
    seed=cfg.training.seed + 100,
)

# Instancia o treinador (sessão inicial: limite de 10 passos efêmeros)
trainer = Trainer(
    config=cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    persistence_backend=backend,
    session_max_steps=10,
)

print(f"Iniciando treinamento no dispositivo '{cfg.device}' (teto de sessão: 10 passos)...")
summary = trainer.run()
print("Treinamento inicial finalizado com sucesso!")
print(f"Global Step Atingido: {summary['global_step']}")
print(f"Perda Final: {summary.get('final_loss', 'N/A')}")

## 6. Auditoria de Checkpoints Persistidos no Drive

Verificação da presença do marcador atômico `COMPLETE.json`, do arquivo de ponteiro `latest.json` e dos hashes SHA-256.

In [ ]:
# [Célula 6] Auditoria de Checkpoints no Repositório Persistente
pers_root = trainer.persistent_run_root
valid_checkpoints = backend.list_complete_checkpoints(pers_root)
print(f"Checkpoints completos e válidos encontrados: {valid_checkpoints}")

latest_dir = backend.resolve_checkpoint(pers_root, selector="latest")
print(f"Diretório resolvido pelo ponteiro latest: {latest_dir.name}")

# Confere arquivos obrigatórios no checkpoint resolvido
assert (latest_dir / "checkpoint.pt").exists(), "checkpoint.pt ausente!"
assert (latest_dir / "manifest.json").exists(), "manifest.json ausente!"
assert (latest_dir / "COMPLETE.json").exists(), "COMPLETE.json ausente!"
print("Integridade física de todos os arquivos de checkpoint confirmada!")

Este bloco cria um segundo objeto `Trainer` no mesmo processo para testar a API de restauração. Isso **não** simula perda da VM nem conclui G5. Para evidência G5, reinicie o runtime Colab, reabra o commit fixado, monte o mesmo Drive e execute apenas as células de preparação e restauração usando a mesma run persistida.


In [ ]:
# [Célula 7] Restauração e Continuidade em Nova Sessão
print("--- Simulando Novo Runtime Efêmero ---")

resumed_cfg = TrainingConfig.from_yaml("configs/training/pilot.yaml")
resumed_cfg.device = cfg.device
resumed_cfg.persistence.backend = "drive"
resumed_cfg.persistence.runs_dir = cfg.persistence.runs_dir
resumed_cfg.training.max_steps = 20  # Mantém idêntica configuração científica

resumed_trainer = Trainer(
    config=resumed_cfg,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    persistence_backend=backend,
    session_max_steps=15,  # Executará mais 5 passos além dos 10 anteriores
)

# Restaura exatamente a partir de 'latest'
resumed_trainer.resume_from_checkpoint(selector="latest")
print(f"Estado restaurado com sucesso! Global Step inicial: {resumed_trainer.global_step}")
assert resumed_trainer.global_step == 10, f"Passo incorreto: {resumed_trainer.global_step}"

# Executa segmento pós-retomada
resumed_summary = resumed_trainer.run()
print(f"Retomada concluída com sucesso! Global Step final: {resumed_summary['global_step']}")
assert resumed_summary['global_step'] == 15, "O loop de treino não atingiu a meta planejada pós-retomada!"
print("Smoke de restauração no mesmo processo concluído; G5 segue pendente até recuperação em runtime Colab novo.")